In [ ]:
import pandas as pd

csv_path = "../../../../../results/results.csv"
df = pd.read_csv(csv_path)

In [11]:
def agent_type_name(t):
    t = str(t).upper()
    if t in ["GROUND", "G"]:
        return "G"
    elif t in ["AERIAL", "A"]:
        return "A"
    else:
        return t

def method_type_name(t):
    t = str(t)
    if "het" in t:
        return "Het"
    elif "hom" in t:
        return "Hom"
    else:
        return t

In [12]:
df["agent_type_norm"] = df["agent_type"].apply(agent_type_name)
df["method_norm"] = df["method"].apply(method_type_name)

In [13]:
# Count how many of each agent type per experiment
agent_count = {}
# Choose one baseline to avoid doubles
baseline = df["method_norm"].unique()[0]
for exp_id, agent_type, method in zip(df["exp_id"], df["agent_type_norm"], df["method_norm"]):
    if exp_id not in agent_count:
        agent_count[exp_id] = ""
    if  method == baseline:
        agent_count[exp_id] += f"{agent_type},"

# Add agent_config to each experiment
df["agent_config"] = [agent_count[exp_id] for exp_id in df["exp_id"]]

In [ ]:
def generate_latex_summary(df):
    agg_cols = ["distance", "time", "traversability", "collision"]

    # Number of significant decimals for each metric
    decimals_avg_max = [0, 0, 2, 2] 
    decimals_std = [0, 0, 2, 2]     

    # Aggregate mean, std, min, max
    summary = (
        df.groupby(["map", "method_norm", "agent_config"])[agg_cols]
        .agg(["mean", "std", "min", "max"])
        .reset_index()
    )

    # Flatten MultiIndex columns
    summary.columns = [f"{col[0]}_{col[1]}" if col[1] else col[0] for col in summary.columns.values]

    # Round numeric columns per metric
    for i, m in enumerate(agg_cols):
        for suffix, dec in zip(["mean", "std", "min", "max"], [decimals_avg_max[i], decimals_std[i], decimals_avg_max[i], decimals_avg_max[i]]):
            col_name = f"{m}_{suffix}"
            if dec == 0:
                summary[col_name] = summary[col_name].round(0).astype(int)
            else:
                summary[col_name] = summary[col_name].round(dec)

    # Create LaTeX-formatted strings
    for i, m in enumerate(agg_cols):
        summary[f"{m}_latex"] = summary.apply(
            lambda row: f"{row[f'{m}_max']}",
            axis=1
        )

    # Keep only relevant columns
    cols = ["map", "method_norm", "agent_config"] + [f"{m}_latex" for m in agg_cols]
    df_disp = summary[cols].copy()

    # Define Explicit Categorical Orders
    map_order = ["forest", "orchard", "park", "cave"]
    # Get agent_config order from the actual data (matching the logic used in the header)
    agent_order = sorted(df_disp["agent_config"].unique(), reverse=True) 
    method_order = df_disp["method_norm"].unique()[::-1]

    df_disp["map"] = pd.Categorical(df_disp["map"], categories=map_order, ordered=True)
    df_disp["agent_config"] = pd.Categorical(df_disp["agent_config"], categories=agent_order, ordered=True)

    # Print header for LaTeX table
    # This ensures the header construction exactly matches the pivot logic
    header_cols = ["Metric", "Method"] + [f"{m}-{a}" for m in map_order for a in agent_order]
    header = " & ".join(header_cols) + " \\\\"
    print(header)

    # Produce LaTeX-friendly row outputs per metric
    for metric in agg_cols:
        print(f"\n% ===== {metric} =====")
        
        pivot = (
            df_disp.pivot_table(
                index=["method_norm"],
                columns=["map", "agent_config"],
                values=f"{metric}_latex",
                aggfunc="first",
                observed=True # Keeps the categorical ordering
            )
            .reindex(index=method_order)
            # Reindex columns to match the exact order of map_order and agent_order
            .reindex(columns=pd.MultiIndex.from_product([map_order, agent_order]))
        )
        
        # Print LaTeX rows
        for method, row in pivot.iterrows():
            values = " & ".join(row.fillna("").astype(str).tolist())
            print(f"{metric} & {method} & {values} \\\\")

generate_latex_summary(df)

Metric & Method & forest-A,G, & forest-A,A,G,G, & orchard-A,G, & orchard-A,A,G,G, & park-A,G, & park-A,A,G,G, & cave-A,G, & cave-A,A,G,G, \\

% ===== distance =====
distance & Hom & 80 & 45 & 86 & 43 & 148 & 75 & 1542 & 696 \\
distance & Het & 115 & 66 & 116 & 63 & 233 & 120 & 1989 & 1195 \\

% ===== time =====
time & Hom & 160 & 90 & 172 & 86 & 295 & 149 & 3083 & 1391 \\
time & Het & 115 & 68 & 163 & 75 & 237 & 126 & 2098 & 1195 \\

% ===== traversability =====
traversability & Hom & 0.16 & 0.24 & 0.61 & 0.4 & 0.91 & 0.68 & 1.0 & 0.97 \\
traversability & Het & 0.07 & 0.1 & 0.0 & 0.0 & 0.11 & 0.11 & 0.04 & 0.01 \\

% ===== collision =====
collision & Hom & 0.37 & 0.24 & 0.67 & 0.63 & 0.45 & 0.22 & 0.7 & 0.64 \\
collision & Het & 0.05 & 0.02 & 0.29 & 0.12 & 0.12 & 0.06 & 0.12 & 0.09 \\
